# MicroDuck Omnidirectional Walking RL: A Beginner Lab

**Version:** v2.4 · **Estimated time:** ~70 min (fundamentals ~35 min + GPU practice ~15 min + comparison)

> This tutorial does not assume prior reinforcement learning or robotics background. Build intuition first, then run the code. Equations are used only to explain key ideas.

---

## A. What problem are we solving?

Give MicroDuck a desired velocity, for example “0.3 m/s forward, 0.1 m/s left, while turning counterclockwise,” and let it decide how all 14 joints should move at each moment so that it can:

- Track forward/back, left/right, and yaw commands as accurately as possible;
- Stay upright, with fewer falls and less slipping;
- Produce continuous, natural gaits without frequent self-collisions or joint-limit hits;
- Keep walking despite sensor noise, actuation delay, mass variation, and external pushes.

Hand-writing control rules that cover every velocity combination for 14 joints is hard. **Reinforcement learning (RL)** lets the robot try actions in simulation, judge outcomes from reward, and gradually learn a control policy from observations to actions.

### After this lesson you should be able to answer

1. What is reinforcement learning, and why is it used for robot control?
2. What are an MDP, observation, action, reward, policy, and episode?
3. What common RL algorithms exist, and why does this lab use PPO?
4. How does MicroDuck’s reward function express “track well, stand stably, and move reasonably” at the same time?
5. How does domain randomization help a simulated policy cope with real-world uncertainty?
6. What do ROCm, PyTorch, UniLab, `microduck_rl_unilab`, and this lab each do?
7. How do you train, replay, and decide whether a gait policy is improving?

### Three-stage practice

| Phase | Config | What you do | Time |
|---|---|---|---|
| **Smoke test** | 4 env × 2 iter | Inspect the full log and confirm the stack runs | ~1 min |
| **1** | 500 env × 300 iter | Train from scratch and render a **10-second** replay | ~5 + 2 min |
| **2** | 2048 env × 500 iter | Replay the bundled demo checkpoint | ~2 min |
| **3** | Compare | Compare reward, episode length, curves, and video | ~10 min |

**Optional:** Continue training from the demo checkpoint for +500 iter (2048 env, ~10 min).

> ⚠️ **Do not Run All.** Run cells in order. A long training cell sitting at `[*]` for several minutes is normal.


## B. Software stack and environment check

### B1 How do the pieces fit together?

```text
┌─────────────────────────────────────────────────────────────────────┐
│ microduck_rl_tutorial (this course)                                 │
│ Notebooks, run scripts, pretrained checkpoint, lab notes            │
└──────────────────────────────┬──────────────────────────────────────┘
                               │ calls
┌──────────────────────────────▼──────────────────────────────────────┐
│ microduck_rl_unilab (MicroDuck task pack)                           │
│ Robot assets, obs/action defs, rewards, domain rand, PPO configs    │
└──────────────────────────────┬──────────────────────────────────────┘
                               │ registers tasks into
┌──────────────────────────────▼──────────────────────────────────────┐
│ UniLab (general robot learning framework)                           │
│ Parallel envs, MuJoCo, sampling, PPO, logs and checkpoints          │
└───────────────┬───────────────────────────────────┬─────────────────┘
                │ simulation                        │ neural training
        ┌───────▼───────────────┐         ┌─────────▼─────────────┐
        │ MuJoCo                │         │ PyTorch               │
        │ dynamics, contacts,   │         │ tensors, autograd,    │
        │ sensors               │         │ PPO updates           │
        └───────────────────────┘         └───────────┬───────────┘
                                                      │ AMD GPU backend
                                          ┌───────────▼───────────┐
                                          │ ROCm / HIP            │
                                          │ AMD GPU compute stack │
                                          └───────────────────────┘
```

### B1.1 Open-source projects this tutorial depends on

| Project | Repo | Role | How this tutorial uses it |
|---|---|---|---|
| **UniLab** | [`Motphys/UniLab`](https://github.com/Motphys/UniLab) | General robot RL framework: unified env APIs, parallel sampling, algorithms, logging, checkpoints, visualization, and multiple sim/compute backends | Creates MuJoCo envs, runs PPO rollouts/updates, saves training artifacts, and provides eval plus interactive playback |
| **microduck_rl_unilab** | [`rocPAI-Forge/microduck_rl_unilab`](https://github.com/rocPAI-Forge/microduck_rl_unilab) | MicroDuck task pack on UniLab: robot assets, BAM actuators, obs/action defs, rewards, commands, domain randomization, curriculum, and task configs | Supplies `microduck_velocity_flat`, 61-D actor observations, 14-D actions, and the training contract behind the reference demo |

Both projects use the Apache-2.0 license. Think of them as: **UniLab is the generic training base; `microduck_rl_unilab` implements MicroDuck’s concrete learning task on that base.** This repository sits on top and organizes the teaching flow and pretrained assets.

Easy mix-ups:

- **ROCm is not an RL framework**: it lets PyTorch run on AMD GPUs.
- **PyTorch does not contain the MicroDuck task**: it handles networks and optimization.
- **UniLab is not the robot model**: it provides generic training and simulation management.
- **`microduck_rl_unilab` defines this lab’s robot, rewards, and task config.**
- **`microduck_rl_tutorial` is the teaching layer**: it does not reimplement PPO; it organizes experiments and notebooks.
- This lab uses `--sim mujoco`. MuJoCo computes robot dynamics; PyTorch/ROCm run the policy network and PPO updates. Those jobs are different.

### B2 Locations inside the container

| Component | Form | Location or version |
|---|---|---|
| ROCm + PyTorch | Base image | torch 2.11.0 with a ROCm backend |
| UniLab | Python package | `unilab[mujoco]==1.0.0` |
| `microduck_rl_unilab` | git clone | `/opt/microduck_rl_unilab` (training working directory) |
| This tutorial | Host mount | `/workspace/microduck_rl_tutorial` |
| Phase 2 demo | Bundled here | `examples/velocity_flat_demo/` |

The entrypoint already sets `UNILAB_EXTRA_REGISTRY_PACKAGES` and `MICRODUCK_ROOT` for task registration. Run the check below and confirm **HIP available**, that the task registers, and that the checkpoint exists.

### B3 Hardware compatibility and reference performance

This lab is not limited to MI210. If the current ROCm, PyTorch, and container device mapping support the GPU, it can also run on Instinct GPUs such as MI300X and on Radeon GPUs supported by the matching ROCm release. Memory, architecture support, and throughput differ by model—trust the self-check, and scale `num_envs` to fit VRAM.

The times below are **measured references on a single MI210**, not a hardware requirement. Training duration and feasible parallel env counts will change on other GPUs.

| | Phase 1 | Phase 2 demo |
|---|---|---|
| Config | **500 × 300** | **2048 × 500** |
| Samples / iter | 500×24 = 12,000 | 2048×24 = 49,152 |
| Total env steps | ~3.6M | ~24.6M |
| Wall clock | ~5 min | ~10 min |
| Final reward | Medium (seed-dependent) | **~94** |
| Episode length | Usually < 500 | **~888 / 1000** |

`2048 env` does not mean each iteration is faster; it means more experience is collected in parallel. The policy interface is: actor observation **61-D**, critic observation **76-D**, action **14-D**.


In [ ]:
!bash /workspace/microduck_rl_tutorial/scripts/check_env.sh

## C. Reinforcement learning and MDPs: one interaction first

### C1 What is reinforcement learning?

RL studies this: **an agent takes actions in an environment and uses feedback to improve long-term behavior.** Unlike supervised learning, the training data usually has no “correct 14-joint answer” for a state—only a reward after the action. The robot must discover better action sequences by trial and error.

```text
                 ┌──────── reward rt, next state st+1 ────────┐
                 │                                            │
env / MicroDuck  ├─ obs ot → policy (network) → action at ────┤
   MuJoCo        │                                            │
                 └──────── repeat until fall or timeout ──────┘
```

Why does RL fit robot tasks?

- Joints are strongly coupled; one action keeps affecting later states;
- There is more than one goal: speed, balance, smoothness, energy, and safety must be traded off;
- Simulation can generate millions of interactions in parallel, far cheaper than hardware trial-and-error;
- A neural policy can learn nonlinear feedback that is hard to write by hand.

RL is not a silver bullet: a poorly designed reward can be exploited, sim and hardware differ, and training is stochastic. That is why we use replay, metrics, domain randomization, and eventual on-robot safety checks.

### C2 What is an MDP?

RL is usually described as a **Markov Decision Process (MDP)**. An MDP is often written `(S, A, P, R, γ)`:

| Symbol | Meaning | MicroDuck example |
|---|---|---|
| `S` state | True situation of the world | Pose, joints, velocities, contacts, … |
| `A` action | Decision the agent can make | 14-D joint targets |
| `P` transition | How an action leads to the next state | MuJoCo dynamics, contacts, actuator model |
| `R` reward | How good the current outcome is | Sum of tracking, upright, gait, … terms |
| `γ` discount | How much future reward is valued | Closer to 1 means more long-horizon stability |

**Markov** intuition: given the current state and action, predicting the next step does not require the full history. A real robot usually sees noisy, delayed, incomplete observations `oₜ`, so code distinguishes “state” from “observation.”

### C3 Mapping onto this task

- **Command:** periodically sample `twist = (vx, vy, yaw_rate)`; ranges are about ±0.4 m/s forward/back, ±0.3 m/s lateral, ±1.0 rad/s yaw.
- **Observation:** body angular velocity, gravity direction, joint position/velocity, last action, velocity command, head command, and so on. The actor uses a 61-D deployable observation.
- **Action:** every 20 ms the policy outputs 14-D joint targets; a BAM voltage servo model turns them into actuator effort.
- **Rates:** physics at 200 Hz (`sim_dt=0.005 s`), policy at 50 Hz (`ctrl_dt=0.02 s`).
- **Episode:** at most 20 s, i.e. 1000 control steps; tilt beyond ~70°, NaNs, or timeout terminate.
- **Policy:** a neural net `πθ(a|o)` with parameters `θ`. A finished `model_*.pt` stores the learned parameters.


## D. From common algorithms to PPO

### D1 Typical RL algorithms

| Family | Examples | Idea | Typical traits |
|---|---|---|---|
| Value-based | DQN | Learn action values, then pick the highest | Classic DQN fits discrete actions, not 14-D continuous joints |
| Policy gradient | REINFORCE | Directly raise the probability of high-return actions | Simple, but high-variance and sample-inefficient |
| Actor-Critic | A2C/A3C | Actor chooses actions; critic evaluates states/actions | Learns policy and value together; lowers policy-gradient variance |
| On-policy actor-critic | **PPO** | Train on data just collected by the current policy, with limited update size | Stable, mature implementations, scales to many parallel envs |
| Off-policy actor-critic | SAC, TD3 | Reuse experience from a replay buffer | High sample reuse for continuous control; different tuning/system cost at large scale |

“On-policy” means training mainly uses data from the current policy; “off-policy” can reuse older-policy data. No algorithm is best for every task.

### D2 Why PPO here?

MicroDuck has 14-D continuous actions, and simulation can run hundreds to thousands of envs at once. PPO matches that:

1. **Fits continuous control:** the policy can output a continuous distribution per joint.
2. **More stable training:** PPO clip limits how far the new policy may move, so one update is less likely to destroy a learned gait.
3. **Easy parallel sampling:** 500 or 2048 envs make a large on-policy batch.
4. **Mature locomotion recipes:** many verified PPO setups exist for robot walking.
5. **Actor-critic structure:** the actor learns actions; the critic estimates future return so we can tell whether an action was better than expected.

PPO’s core is not “always pick the max-reward action.” It loops:

```text
┌─→ sample in parallel with the current policy
│        │
│        ▼
│   compute return and advantage (GAE)
│        │
│        ▼
│   mini-batch updates for Actor and Critic
│        │
└────────┘
```

- **Return:** discounted reward accumulated from the current time.
- **Value:** the critic’s prediction of future return.
- **Advantage:** how much better the outcome was than the critic expected.
- **PPO clip:** if the new policy changes too much vs the old one, truncate that optimization gain—“don’t step too far at once.”
- **Entropy:** keep some exploration so the policy does not become fully deterministic too early.

This task uses an **asymmetric actor-critic**: the actor sees only the deployable 61-D noisy observation; during training the critic also sees foot contacts, foot height, and body linear velocity (76-D total). Extra information helps the critic evaluate training data more accurately, but deployment does not need those privileged observations.

<details>
<summary>PPO hyperparameters used here (no need to memorize as a beginner)</summary>

- Actor/Critic MLP: `[512, 256, 128]`, ELU
- PPO clip: `0.2`; discount `gamma=0.99`; GAE `lambda=0.95`
- 5 epochs per batch, split into 4 mini-batches
- Initial action-noise std `1.0`, entropy coefficient `0.01`
- Adaptive learning rate, target KL divergence `0.01`
- Left-right symmetric mirror loss, coefficient `0.5`

Together these affect update stability, exploration, and compute. Do not casually change many of them in one experiment.

</details>

### D3 What does one iteration do?

1. Each parallel env runs the current policy for **24 control steps**;
2. Record `(observation, action, reward, terminated, value estimate)`;
3. Gather all transitions, e.g. `500 × 24 = 12,000` per round in phase 1;
4. Compute advantage and update Actor/Critic on the GPU;
5. Write logs, save a checkpoint if needed, then start the next round.

| Hydra parameter | Meaning |
|---|---|
| `algo.num_envs` | Independent simulations running at once |
| `algo.max_iterations` | How many sample-and-update loops |
| `algo.save_interval` | Save a checkpoint every N iterations |
| `training.no_play=true` | Do not auto-replay after training |
| `training.play_steps` | How many 50 Hz control steps to run at eval |
| `training.log_root` | Root directory for logs and checkpoints |


## E. Reward, domain randomization, and curriculum

### E1 What is a reward function?

A reward function turns “how we want the robot to behave” into a scalar at every step. The policy maximizes long-horizon return:

`total reward = task reward + posture/gait reward − cost of unsafe or jerky behavior`

Reward is not a human demonstration and not a success rate. A value of 94 does not mean 94% correct. It is comparable only under the **same task and same reward definition**.

Omnidirectional MicroDuck walking cannot reward forward speed alone, or the robot could score by falling and sliding. The real task uses multiple objectives:

| Design goal | Representative term | Weight direction | What it prevents |
|---|---|---|---|
| Track planar velocity | `tracking_lin_vel` | `+2.0` | Ignoring `(vx, vy)` or only one direction |
| Track yaw rate | `tracking_ang_vel` | `+2.0` | Not turning with yaw-rate |
| Stay upright | `upright` | `+2.0` | Tilting or lying down to “farm speed” |
| Track head pose | `head_pose_tracking` | `+2.0` | Body walks while the head is uncontrolled |
| Reasonable leg pose | `leg_pose` | `+1.0` | Bizarre joint configurations |
| Reasonable air time | `air_time` | `+3.0` | Dragging feet or no real gait |
| Foot height / slip | `foot_clearance -2.0`, `foot_swing_height -0.25`, `foot_slip -0.1` | penalty | Too high/low swing, sliding on contact |
| Safety and smoothness | self-collision `-1.0`, joint limits `-1.0`, action change `-0.1` | penalty | Hitting itself, slamming limits, jitter |
| Body stability | body angular velocity `-0.05`, angular momentum `-0.02` | penalty | Violent upper-body motion |

Reward design is a multi-objective trade-off: too much tracking can yield a brutal gait; too much smoothness can make the robot refuse to move. Look at total reward, per-term rewards, episode length, **and** video—not a single number.

### E2 What is domain randomization?

Mass, friction, sensors, and actuators in simulation are more ideal than on hardware. **Domain randomization** randomly changes those conditions per environment during training so the policy works for a family of possible worlds, not one exact parameter set.

Examples in this task:

- Observation noise and IMU bias; ~20 ms joint-velocity delay;
- Battery voltage, load sag, command delay, and actuator friction variation;
- Robot mass/inertia, CoM, joint initialization, and foot friction;
- Random pushes every 3–6 s to train disturbance recovery.

The main point is **robustness** and a smaller sim-to-real gap. It does not erase model error and does not replace hardware tests, action limits, or e-stop.

### E3 Domain randomization is not curriculum

- **Domain randomization** answers “what random world is each env?”
- **Curriculum** answers “when does difficulty increase?”

This task gradually strengthens action-change penalties, widens head-command/CoM randomization, and raises the fraction of stand-still commands. Curriculum advances by env control steps. Because each PPO iteration rollouts 24 steps, `12000 steps` in the config is about `500 iterations`. Raising `num_envs` increases samples per round; it does not by itself make a single env’s curriculum clock tick faster.


## F. Smoke test (4 env × 2 iter, ~1 min)

This step runs only 2 iterations to verify ROCm/PyTorch, task registration, MuJoCo XML, sampling, PPO updates, and checkpoint writes.

We **intentionally keep the raw full log** so first-time students can see the output structure. Watch for:

- `Learning iteration 0/2`: current training round;
- `Mean reward`: average cumulative reward of recent episodes—low early on is normal;
- `Mean episode length`: average surviving control steps; very short usually means frequent falls;
- `Collection time / Learning time`: sampling vs network-update time;
- `ETA`: remaining time at the current pace.

> The smoke test only proves “it runs.” Two iterations cannot learn a stable walk.


In [ ]:
%%bash
set -euo pipefail
cd "${MICRODUCK_ROOT:-/opt/microduck_rl_unilab}"

# Only 2 iter: keep the full raw microduck-train output so log fields are visible.
microduck-train --algo ppo --task microduck_velocity_flat --sim mujoco \
  algo.num_envs=4 algo.max_iterations=2 training.no_play=true \
  training.logger=tensorboard training.log_root=/workspace/runs


---

# Phase 1: 500 env × 300 iter

**Goal:** complete a medium-scale PPO run yourself and observe a policy that is “learning, but not fully trained.”

| Parameter | Value | Beginner reading |
|---|---|---|
| `algo.num_envs` | 500 | 500 independent MicroDuck simulations at once |
| `algo.max_iterations` | 300 | Repeat 300 times: “sample 24 steps → update the net” |
| `algo.save_interval` | 50 | Save every 50 iter so a crash does not wipe everything |
| `LAB_LOG_EVERY` | 10 | Notebook refreshes a summary every 10 iter |

Full training output goes to `/workspace/runs/logs/phase1_500x300.log`; the notebook keeps a single concise status refresh. Watch the overall trend of reward and episode length; they will not rise monotonically.

⏱ Reference time: about **5 min** on a single MI210 in this tutorial. Other compatible AMD GPUs will differ by model, ROCm/PyTorch version, and `num_envs`. A cell at `[*]` with a busy GPU still means training is running.


In [ ]:
%%bash
set -euo pipefail
export LAB_TRAIN_LOG=/workspace/runs/logs/phase1_500x300.log
export LAB_LOG_EVERY=10
bash "${LAB_ROOT:-/workspace/microduck_rl_tutorial}/scripts/run_train.sh" \
  --algo ppo --task microduck_velocity_flat --sim mujoco \
  algo.num_envs=500 algo.max_iterations=300 algo.save_interval=50 \
  training.no_play=true training.logger=tensorboard training.log_root=/workspace/runs


### How to read the one-line log after training

Example:

```text
[train] iter  299/300 | reward    4.59 | ep  78.59 |  0:03:59 | ETA  0:00:00 | 0.80s/iter
```

| Field | Meaning | How to read this run |
|---|---|---|
| `iter 299/300` | PPO iterations are 0-indexed; `299` is the 300th sample+update | Planned 300 iter finished |
| `reward 4.59` | Mean cumulative reward of recently finished episodes; tracking, stability, smoothness, energy, etc. combined—not a 0–100 score | Compare only with the same task and reward |
| `ep 78.59` | Mean episode length: control steps before fall or timeout | At 50 Hz this is `78.59 ÷ 50 ≈ 1.57 s`, so episodes still end early |
| `0:03:59` | Wall-clock time since training started | About 4 minutes here |
| `ETA 0:00:00` | Remaining time at the recent pace | Almost done |
| `0.80s/iter` | Average time per PPO iteration | Runtime efficiency, not policy quality |

#### If reward keeps going up, is training necessarily getting better?

**A rising long-term trend is a good sign, but a single bump or short-term noise does not prove the policy improved.** PPO samples randomly, so reward is not monotonic. It can also dip when curriculum gets harder, or look high because of reward hacking.

Judge training with all of:

1. **Reward trend:** tens of iterations, not the last point;
2. **Episode length:** whether it approaches 1000 steps (~20 s cap on this task);
3. **Eval video:** upright, tracking commands, without obvious slip, jitter, or heading error.

This run’s `reward=4.59, ep=78.59` means the 300-iter policy learned *something*, but episodes end after ~1.57 s, so it is still a weak early policy. Next we compare it with the pretrained demo at `ep≈888` (~17.8 s).

> Mini exercise: if the next run gets `ep=750`, convert that to seconds first, then use the reward curve and video to decide whether it really improved.


### Phase 1 · Eval: render a 10-second replay

Training vs eval: training explores and updates parameters; eval loads a checkpoint, runs the current policy, and renders video without learning.

| Parameter | Meaning |
|---|---|
| `--load-run -1` | Load the latest training run for this task |
| `training.play_steps=500` | Control rate is 50 Hz, so `500 × 0.02 s = 10 s` |

Ten seconds makes it easier to see: whether it stays up, how it reacts after command changes, left/right gait symmetry, foot slip, and whether failures are rare or persistent.


### Bundled reference clip (phase 1)

The next cell plays [`notebooks/assets/_phase1.mp4`](../assets/_phase1.mp4), a typical 300-iter eval shared by the Chinese and English notebooks. Watch this first, then run the following cell to generate your own eval video.


In [ ]:
from pathlib import Path
from IPython.display import Video, display

candidates = [
    Path("/workspace/microduck_rl_tutorial/notebooks/assets/_phase1.mp4"),
    Path("..") / "assets" / "_phase1.mp4",
]
video = next((p.resolve() for p in candidates if p.is_file()), None)
if video is None:
    raise FileNotFoundError(
        "Missing teaching clip notebooks/assets/_phase1.mp4"
    )
print(f"Phase 1 bundled clip: {video} ({video.stat().st_size / 1024:.0f} KiB)")
display(Video(
    data=video.read_bytes(),
    embed=True,
    mimetype="video/mp4",
    html_attributes='controls autoplay loop muted playsinline style="max-width:100%;height:auto"',
))


In [ ]:
# Phase 1 eval: show the 10-second robot replay in this cell when it finishes
import os
import subprocess
from pathlib import Path

from IPython.display import Video, display

microduck_root = os.environ.get("MICRODUCK_ROOT", "/opt/microduck_rl_unilab")
log_root = Path("/workspace/runs/MicroduckVelocityFlat")

subprocess.run(
    [
        "microduck-eval", "--algo", "ppo",
        "--task", "microduck_velocity_flat", "--sim", "mujoco",
        "--load-run", "-1",
        "training.play_steps=500", "training.log_root=/workspace/runs",
    ],
    cwd=microduck_root,
    check=True,
)

videos = sorted(log_root.glob("*_mujoco/play_video.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise FileNotFoundError("Eval finished, but play_video.mp4 was not found. Check the logs above.")

video = videos[-1]
print(f"\nRobot playback: {video} ({video.stat().st_size / 1024:.0f} KiB)")
display(Video(
    data=video.read_bytes(),
    embed=True,
    mimetype="video/mp4",
    html_attributes='controls autoplay loop muted playsinline style="max-width:100%;height:auto"',
))


In [ ]:
# Phase 1: read the training summary (robot video was shown in the previous eval cell)
import json
from pathlib import Path

LOG = Path("/workspace/runs/MicroduckVelocityFlat")
runs = sorted(LOG.glob("*_mujoco"), key=lambda p: p.stat().st_mtime)
if not runs:
    raise FileNotFoundError("Please finish phase 1 training first.")
PHASE1_RUN = runs[-1]

s1 = json.loads((PHASE1_RUN / "run_summary.json").read_text())
print(f"Phase 1 run: {PHASE1_RUN.name}")
print(f"  reward={s1['final_mean_reward']:.2f}  ep_len={s1['mean_episode_length']:.1f}  "
      f"iters={s1['completed_iterations']}")


### How to read the phase 1 replay: why do many robots barely move?

The phase 1 video renders several environments, but they all use the **same 300-iter policy**. Robots can differ in initial state, velocity command, and randomization. One robot falling while another stands for a while does **not** mean different models were loaded.

Separate two cases while watching:

- If the target speed is near 0, standing or balancing in place can be the correct response;
- If there is a clear motion command but the robot stays still, only steps in place, or falls quickly, the policy has not yet formed a reliable command-tracking gait.

This run’s `reward≈4.59` and `episode length≈78.59 steps` (~1.57 s at 50 Hz), plus a video of “most barely move, a few fall,” means **this particular 300-iter checkpoint is still an under-converged early policy**. Ten seconds is the length of the eval clip, not each episode’s lifetime; robots may terminate and reset several times in the video.

That also sets expectations for later teleop: keyboard commands can reach the policy, but this checkpoint may show little gait, drift, or fall after command changes. Treat that as limited model skill first, not a keyboard, network, or inference-stack failure.

> “300 iter” is not universally “undertrained.” The conclusion is only for this reward, episode length, and video together. `02_interactive_teleop.ipynb` will first verify the stack with the stable demo, then load this student model for contrast.


---

# Phase 2: load the pretrained reference demo

**Goal:** skip a long specialized training wait and load the bundled checkpoint to watch a reference policy that passed teleop-oriented behavior checks.

- A **checkpoint** is model parameters saved at some training moment; here it is `examples/velocity_flat_demo/model_950.pt`.
- This model used 2048 parallel envs and continued directional fine-tuning from a stable 500-iter baseline. It is not a simple “950 is always better than 500.”
- Selection is not based on final reward, but on 32 trials × 10 s of fixed commands: forward, back, left, and right all passed; yaw improved but remains weaker than translation.
- Full recipe and six-direction results: `examples/velocity_flat_demo/run_summary.json` and `cardinal_eval*.json`.

| Reference demo metric (training performance on MI210) | Value | How to read it |
|---|---|---|
| Linear directions | 4 / 4 passed | Stability, sign, error, and cross-axis drift together |
| seed-123 mean score | ~81.9 / 100 | Six-direction composite, not PPO reward |
| seed-123 worst direction | ~59.4 / 100 | Current bottleneck is still yaw |
| Yaw survival | 100% | No fall in 10 s; does **not** mean yaw magnitude is fully accurate |
| Checkpoint | `model_950.pt` | Chosen by an automatic checkpoint sweep, not simply the final file |


### Bundled reference clip (phase 2)

The next cell plays [`notebooks/assets/_phase2.mp4`](../assets/_phase2.mp4), a typical Reference Demo replay shared by both language notebooks.


In [ ]:
from pathlib import Path
from IPython.display import Video, display

candidates = [
    Path("/workspace/microduck_rl_tutorial/notebooks/assets/_phase2.mp4"),
    Path("..") / "assets" / "_phase2.mp4",
]
video = next((p.resolve() for p in candidates if p.is_file()), None)
if video is None:
    raise FileNotFoundError(
        "Missing teaching clip notebooks/assets/_phase2.mp4"
    )
print(f"Phase 2 bundled clip: {video} ({video.stat().st_size / 1024:.0f} KiB)")
display(Video(
    data=video.read_bytes(),
    embed=True,
    mimetype="video/mp4",
    html_attributes='controls autoplay loop muted playsinline style="max-width:100%;height:auto"',
))


In [ ]:
# Phase 2 demo eval: show robot playback in this cell when it finishes
import os
import subprocess
from pathlib import Path

from IPython.display import Video, display

microduck_root = os.environ.get("MICRODUCK_ROOT", "/opt/microduck_rl_unilab")
demo = Path("/workspace/microduck_rl_tutorial/examples/velocity_flat_demo")

subprocess.run(
    [
        "microduck-eval", "--algo", "ppo",
        "--task", "microduck_velocity_flat", "--sim", "mujoco",
        "--load-run", str(demo),
        "training.play_steps=200", "training.log_root=/workspace/runs",
    ],
    cwd=microduck_root,
    check=True,
)

video = demo / "play_video.mp4"
if not video.is_file():
    raise FileNotFoundError("Demo eval finished, but play_video.mp4 was not found. Check the logs above.")

print(f"\nRobot playback: {video} ({video.stat().st_size / 1024:.0f} KiB)")
display(Video(
    data=video.read_bytes(),
    embed=True,
    mimetype="video/mp4",
    html_attributes='controls autoplay loop muted playsinline style="max-width:100%;height:auto"',
))


In [ ]:
# Phase 2: read the reference-demo selection summary (video was shown in the previous eval cell)
import json
from pathlib import Path

DEMO = Path("/workspace/microduck_rl_tutorial/examples/velocity_flat_demo")
s2 = json.loads((DEMO / "run_summary.json").read_text())
print("=== Phase 2 reference demo run_summary ===")
print(json.dumps({k: s2[k] for k in (
    "checkpoint", "completed_iterations", "num_envs",
    "selection_eval_yaw_rate", "selection_eval_mean_score",
    "selection_eval_worst_direction_score", "selection_reason",
)}, indent=2, ensure_ascii=False))


---

# Phase 3: how do you tell if the policy got better?

Do not conclude from a single reward number. Compare phase 1 and phase 2 in this order:

1. **Episode length first:** frequent early falls? 1000 steps is the 20 s cap.
2. **Then video:** stay upright? Track forward/lateral/yaw commands? Slip, jitter, one-sided gait?
3. **Then the reward curve:** long-term rise, or violent oscillation/regression? Noise on RL curves is normal.
4. **Then training scale:** phase 1 ~3.6M env steps, phase 2 ~24.6M, about 6.8× apart.
5. **Avoid over-causal claims:** env count and iteration count both changed, so this shows overall scale, not a clean isolation of either factor.

### Expected picture

| Dimension | Phase 1 500×300 | Phase 2 2048×500 |
|---|---|---|
| Transitions / iter | 12,000 | 49,152 |
| Total env steps | ~3.6M | ~24.6M |
| Training endpoint | Stopped at 300 iter | Owner default 500-iter endpoint |
| Typical gait | Still unstable, may terminate early | More stable, usually near a full episode |
| Reward | Seed-dependent; no preset answer | Final mean ~94 |

### Discussion questions

1. Does a longer episode length always mean more accurate velocity tracking? Why?
2. If you only increase survival reward, what “cheating” behavior might the robot learn?
3. 2048 envs collect more data per round—why does that not guarantee shorter wall-clock time?
4. To compare `num_envs` rigorously, how would you set controls and multiple random seeds?
5. Which video symptoms might expose problems in the reward or domain-randomization config?

**TensorBoard (optional)** for scalars vs iteration:

```bash
tensorboard --logdir /workspace/runs/MicroduckVelocityFlat --bind_all
```


<details>
<summary><b>Reference answers (think first, then expand)</b></summary>

### 1. Does longer episode length always mean more accurate velocity tracking?

No. It only means the robot lasted longer before termination. It might stand in place or step slowly to avoid falling without tracking the target speed. Check velocity-tracking reward, episode length, and eval video together.

### 2. What “cheating” can a survival-only reward produce?

The policy may freeze, crouch, stand stiffly, or keep tiny steps, because those behaviors survive without completing the walking task. That is **reward hacking**: optimizing the score we wrote, not the behavior we wanted.

### 3. Why more envs per round does not guarantee shorter wall-clock time?

More parallel envs increase simulation, VRAM, data movement, and PPO-update work. GPU parallelism raises throughput, but after hardware saturates, extra envs can make each iteration slower. Compare `env steps/s`, total env steps, and total wall time—not iteration count alone.

### 4. How to compare `num_envs` rigorously?

Fix task config, network, PPO hyperparameters, hardware, and **total env steps**; when you change `num_envs`, invert iteration count so groups see the same sample budget. Run each group on the same seed set, report mean and std, and compare both learning quality and training throughput. Otherwise you cannot tell `num_envs` from data volume or randomness.

### 5. Which video symptoms may expose config issues?

- Persistent lean to one side: left-right symmetry, command tracking, or randomization may be weak;
- Obvious foot slip: foot-slip penalty, friction, or contact modeling may be off;
- High-frequency jitter: weak action-smoothness penalty, or insufficient delay/noise coverage;
- Only standing, refusing to walk: survival reward too strong vs task reward;
- Frequent falls after command changes: curriculum ramped too fast, or command coverage too narrow;
- Tiny sim changes break it: domain randomization too narrow; if training itself will not converge, the range may be too wide.

These only generate hypotheses; check per-term rewards and repeated evals to confirm.

</details>


In [ ]:
# Phase 3: phase 1 training metrics + reference-demo behavior gate
import json
from pathlib import Path

import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

LOG = Path("/workspace/runs/MicroduckVelocityFlat")
DEMO = Path("/workspace/microduck_rl_tutorial/examples/velocity_flat_demo")

phase1 = sorted(LOG.glob("*_mujoco"), key=lambda p: p.stat().st_mtime)[-1]
s1 = json.loads((phase1 / "run_summary.json").read_text())
s2 = json.loads((DEMO / "run_summary.json").read_text())

print("=== Phase 1: this training run ===")
for name, value in (
    ("num_envs", s1["global_num_envs"]),
    ("iterations", s1["completed_iterations"]),
    ("total_env_steps", s1["total_env_steps"]),
    ("final_reward", f"{s1['final_mean_reward']:.1f}"),
    ("episode_length", f"{s1['mean_episode_length']:.0f}"),
    ("train_wall_sec", f"{s1['training_wall_time_sec']:.0f}"),
):
    print(f"{name:24} {value}")

print("\n=== Phase 2: reference demo fixed-command gate ===")
for name, value in (
    ("checkpoint", s2["checkpoint"]),
    ("training_iterations", s2["completed_iterations"]),
    ("selection_mean_score", f"{s2['selection_eval_mean_score']:.1f}"),
    ("worst_direction_score", f"{s2['selection_eval_worst_direction_score']:.1f}"),
    ("yaw_rate", s2["selection_eval_yaw_rate"]),
):
    print(f"{name:24} {value}")
print("\nNote: PPO reward and the six-direction gate score are different metrics; do not compare them by raw magnitude.")

def plot_scalars(run_dir: Path, tags: list[str]):
    ev = list(run_dir.glob("events.out.tfevents.*"))
    if not ev:
        return None
    ea = EventAccumulator(str(ev[0]))
    ea.Reload()
    out = {}
    for tag in tags:
        if tag in ea.Tags()["scalars"]:
            scalars = ea.Scalars(tag)
            out[tag] = ([x.step for x in scalars], [x.value for x in scalars])
    return out

tags = ["Train/mean_reward", "Train/mean_episode_length"]
data = plot_scalars(phase1, tags)
if data:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, tag in zip(axes, tags):
        if tag in data:
            xs, ys = data[tag]
            ax.plot(xs, ys)
        ax.set_title(tag)
        ax.set_xlabel("iteration")
    fig.suptitle(f"Phase 1 training curves · {phase1.name}")
    plt.tight_layout()
    plt.show()


## J. Recap and glossary

### Done checklist

| ✓ | You have |
|---|---|
| □ | Separated ROCm, PyTorch, UniLab, the MicroDuck task pack, and this lab |
| □ | Understood the RL loop, MDP, policy, reward, and episode |
| □ | Read the full 2-iter smoke-test log |
| □ | Finished phase 1 500×300 train and 10 s eval |
| □ | Replayed the phase 2 2048×500 demo |
| □ | Judged the policy with metrics, curves, and video together |

### Core takeaways

- RL learns a control policy from simulated trial-and-error and reward; it does not need a correct joint action per state.
- PPO is actor-critic with limited policy-update size; it is stable and fits continuous actions plus large parallel sampling, but not every task.
- The reward defines “what good means.” It must cover task, stability, gait, safety, and smoothness, and you must watch for reward hacking.
- `num_envs` sets parallel data per round; `max_iterations` sets how many sample-update loops. Training scale is not either number alone.
- Domain randomization improves robustness under uncertainty; curriculum changes difficulty over time. They are different ideas.
- Good sim performance is not hardware deployment. You still need model checks, latency tests, action limits, fall protection, and e-stop.

### Glossary

| Term | One-line meaning |
|---|---|
| Agent | The learner/decision maker; here, the policy network |
| Environment / env | The simulated world that takes actions and returns the next obs and reward |
| Observation | What the policy can actually read; may be noisy, delayed, incomplete |
| Action | Commands the policy sends to the controller; here, 14-D joint targets |
| Policy / Actor | Model that maps observations to actions |
| Critic / Value | Predicts future return to help the actor learn |
| Reward | Immediate feedback each step; not a success rate |
| Return | Discounted cumulative reward from a given step |
| Episode | One interaction from reset until fall or timeout |
| Rollout | A stretch of transitions collected with the current policy |
| Transition | One `(obs, action, reward, next_obs, done)` record |
| Iteration | One “rollout sampling + PPO update” round |
| Checkpoint | Saved model parameters (and related state) from training |
| Domain randomization | Randomize sim conditions to improve robustness |
| Curriculum | Gradually change task or randomization difficulty |
| Sim-to-real | Transfer a policy learned in simulation onto a real robot |

---

## Appendix (optional): continue +500 iter from the demo

Train **500 more iterations** on the phase 2 demo (2048 env, ~10 min) and see whether it improves:

```bash
cd /opt/microduck_rl_unilab

# Continue training (max_iterations=500 means 500 more iter on the reference checkpoint)
bash /workspace/microduck_rl_tutorial/scripts/run_train.sh \
  --algo ppo --task microduck_velocity_flat --sim mujoco \
  algo.num_envs=2048 algo.max_iterations=500 algo.save_interval=100 \
  algo.load_run=/workspace/microduck_rl_tutorial/examples/velocity_flat_demo \
  training.no_play=true training.logger=tensorboard training.log_root=/workspace/runs

# Eval after continue-training
microduck-eval --algo ppo --task microduck_velocity_flat --sim mujoco \
  --load-run -1 training.play_steps=200 training.log_root=/workspace/runs
```

<details>
<summary>Further reading: GAE · PPO clip · mirror loss · curriculum</summary>

- **GAE:** estimate advantage from multi-step rewards and the critic, trading bias vs variance.
- **PPO clip:** limit optimization gain from the new/old policy probability ratio so one update cannot jump too far.
- **Mirror loss:** constrain the policy with biped left-right symmetry so gaits do not favor one side.
- **Curriculum:** change reward weights, command ranges, or randomization over training; not simply “turn on all domain randomization.”

</details>

---

## Troubleshooting

| Symptom | What to do |
|---|---|
| Training cell stuck at `[*]` | Normal; check whether `runs/` grew a new directory and whether `rocm-smi` shows a busy GPU |
| Eval has no `play_video.mp4` | Confirm `--load-run` points at a path that contains `model_*.pt` |
| No robot picture | Eval is off-screen; wait for the cell to finish—the embedded video plays under the log |
| Browser blocks autoplay | Click the center of the video; it is muted and set to loop |
| `HIP not available` | Confirm compose maps `/dev/kfd` and `/dev/dri` |


---

## Next: Phase 4 · keyboard-driven inference

After training and video eval, open [`02_interactive_teleop.ipynb`](02_interactive_teleop.ipynb):

- Load the stable demo first to verify the teleop stack, then switch to phase 1’s `model_299.pt`;
- Drive a single robot in the browser MuJoCo desktop and compare the two models;
- Use `↑/↓` for forward/back, `←/→` for lateral, `Q/E` for yaw;
- Verify the 61-D observation → PPO actor → 14-D joint-action closed loop at runtime.

This phase validates simulated inference. It is not the same as hardware deployment.

After 01 and 02, open [`03_quiz.ipynb`](03_quiz.ipynb) for five multiple-choice checks on this lesson.
